# Figure 13.20: Common pooling operations used to downsample feature maps

Choose a pooling type and see how the feature map is downsampled.

- **Max pooling**: takes the largest value in each window
- **Average pooling**: takes the mean of all values in each window
- **L2 pooling**: computes the L2 norm √(Σxᵢ²) of each window

All three have **no learnable parameters** — they are fixed aggregation operations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets


def pool_value(vals, pool_type):
    if pool_type == 'max':
        return float(np.max(vals))
    elif pool_type == 'avg':
        return float(np.mean(vals))
    else:  # l2
        return float(np.sqrt(np.sum(np.array(vals) ** 2)))


def draw_pooling(N=6, K=2, S=2, pool_type='max', seed=42):
    rng = np.random.default_rng(seed)
    X = rng.integers(1, 10, size=(N, N)).astype(float)

    raw = (N - K) / S + 1
    if raw != int(raw) or raw <= 0:
        print(f'Invalid config: (N−K)/S = ({N}−{K})/{S} = {raw:.3f} is not an integer')
        return
    O = int(raw)

    Y = np.zeros((O, O))
    for i in range(O):
        for j in range(O):
            window = X[i*S:i*S+K, j*S:j*S+K]
            Y[i, j] = round(pool_value(window.flat, pool_type), 2)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    pool_label = {'max': 'Max', 'avg': 'Average', 'l2': 'L2'}[pool_type]

    # --- Input ---
    ax = axes[0]
    cmap_in = plt.cm.Blues
    im = ax.imshow(X, cmap=cmap_in, aspect='equal')
    for i in range(N):
        for j in range(N):
            ax.text(j, i, str(int(X[i, j])), ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='white' if X[i, j] > X.max() * 0.6 else '#0f172a')
    # Draw pool grid lines
    for t in range(0, N + 1, S):
        ax.axhline(t - 0.5, color='orange', lw=2, alpha=0.7)
        ax.axvline(t - 0.5, color='orange', lw=2, alpha=0.7)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Input Feature Map ({N}×{N})', fontsize=11)

    # --- Pool windows ---
    ax = axes[1]
    colors = plt.cm.Oranges(np.linspace(0.3, 0.8, O * O))
    ax.set_xlim(0, O); ax.set_ylim(0, O)
    for idx, (i, j) in enumerate([(i, j) for i in range(O) for j in range(O)]):
        window = X[i*S:i*S+K, j*S:j*S+K]
        v = round(pool_value(window.flat, pool_type), 2)
        rect = patches.Rectangle((j, O - 1 - i), 1, 1, fc=colors[idx], ec='white', lw=1.5)
        ax.add_patch(rect)
        ax.text(j + 0.5, O - 0.5 - i, str(v), ha='center', va='center',
                fontsize=10, fontweight='bold', color='#0f172a')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect('equal')
    ax.set_title(f'Pool windows → {pool_label} value', fontsize=11)

    # --- Output ---
    ax = axes[2]
    im2 = ax.imshow(Y, cmap=plt.cm.Greens, aspect='equal')
    for i in range(O):
        for j in range(O):
            ax.text(j, i, str(Y[i, j]), ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='white' if Y[i, j] > Y.max() * 0.6 else '#0f172a')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{pool_label} Pooled Output ({O}×{O})', fontsize=11)

    fig.suptitle(
        f'Pool size {K}×{K}, Stride {S}  →  {N}×{N} input  →  {O}×{O} output  (downsampled ~{S}×)',
        fontsize=11, y=0.02
    )
    plt.tight_layout()
    plt.show()


widgets.interact(
    draw_pooling,
    N=widgets.IntSlider(min=4, max=10, step=1, value=6, description='Input N', continuous_update=False),
    K=widgets.IntSlider(min=2, max=4, step=1, value=2, description='Pool size K', continuous_update=False),
    S=widgets.IntSlider(min=1, max=4, step=1, value=2, description='Stride S', continuous_update=False),
    pool_type=widgets.Dropdown(options=[('Max pooling', 'max'), ('Average pooling', 'avg'), ('L2 pooling', 'l2')],
                               value='max', description='Pool type'),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=42, description='Input seed', continuous_update=False),
)